# Build a reusable transaction parser

In [25]:
def enrich_category(text, category):
    text = text.lower()

    if category and category.strip():
        return category

    if "swiggy" in text or "zomato" in text:
        return "food"
    elif "amazon" in text:
        return "shopping"
    elif "uber" in text:
        return "travel"
    elif "salary" in text:
        return "income"
    elif "netflix" in text:
        return "subscription"
    elif "local store" in text:
        return "shopping"

    return "others"

In [27]:
from gliner2 import GLiNER2
extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
schema = {
    "transaction": [
        "amount::money value like ₹500, 1000 INR",
        "merchant::company or service like Amazon, Swiggy, Uber",
        "category::expense type like food, travel, shopping, income, subscription"
    ]
}

def parse_transaction(text):
    try:
        result = extractor.extract_json(text, schema)
        txn_list = result.get("transaction", [])

        if not txn_list:
            return {
                "amount": "unknown",
                "merchant": "unknown",
                "category": enrich_category(text, None)
            }

        txn = txn_list[0]

        amount = txn.get("amount", "unknown")
        merchant = txn.get("merchant", "unknown")

        # IMPORTANT: handle list output
        raw_category = txn.get("category")
        if isinstance(raw_category, list):
            raw_category = raw_category[0] if raw_category else None

        # APPLY enrichment properly
        category = enrich_category(text, raw_category)

        return {
            "amount": amount,
            "merchant": merchant,
            "category": category
        }

    except Exception as e:
        print("ERROR:", e)  # debug visibility
        return {
            "amount": "error",
            "merchant": "error",
            "category": "error"
        }
def clean(value):
    if isinstance(value, list):
        return value[0] if value else "unknown"
    return value

You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


In [28]:
inputs = [
    "Paid ₹500 to Swiggy",
    "Amazon order ₹1200",
    "Uber ride ₹300",
    "Salary credited ₹50000",
    "Netflix subscription ₹499",
    "Paid 200 at local store"
]

for text in inputs:
    print(text)
    print(parse_transaction(text))
    print("-" * 40)

Paid ₹500 to Swiggy
{'amount': ['₹500'], 'merchant': ['Swiggy'], 'category': 'food'}
----------------------------------------
Amazon order ₹1200
{'amount': ['₹1200'], 'merchant': ['Amazon'], 'category': 'shopping'}
----------------------------------------
Uber ride ₹300
{'amount': ['₹300'], 'merchant': ['Uber'], 'category': 'ride'}
----------------------------------------
Salary credited ₹50000
{'amount': 'unknown', 'merchant': 'unknown', 'category': 'income'}
----------------------------------------
Netflix subscription ₹499
{'amount': 'unknown', 'merchant': 'unknown', 'category': 'subscription'}
----------------------------------------
Paid 200 at local store
{'amount': ['200'], 'merchant': ['local store'], 'category': 'shopping'}
----------------------------------------
